## 价格通知书_需商折 处理流程
从  中筛选指定时间范围的数据，按价格通知书号去重，分类商折字段，输出累计结果。

In [ ]:
import pandas as pd
import re
from datetime import datetime
import os

# ===== 请在此设置时间区间 =====
START_DATE = "2026-05-01"   # 起始日期
END_DATE = "2026-05-31"     # 结束日期
TODAY = datetime.now().strftime("%Y/%m/%d")

INPUT_FILE = "../data/eps3_mass.xlsx"
OUTPUT_DIR = "../output/价格通知书_需商折"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "价格通知书_需商折.xlsx")

In [ ]:
# 1. 读取数据
df = pd.read_excel(INPUT_FILE)
print(f"读取总行数: {len(df)}")

# 2. 按审批通过时间筛选
df["审批通过时间"] = pd.to_datetime(df["审批通过时间"], errors="coerce")
mask = (df["审批通过时间"] >= START_DATE) & (df["审批通过时间"] <= END_DATE)
df = df[mask].copy()
print(f"时间筛选 ({START_DATE} ~ {END_DATE}): {len(df)} 行")

In [ ]:
# 3. 按价格通知书号去重
df = df.drop_duplicates(subset=["价格通知书号"], keep="first")
print(f"去重后: {len(df)} 行")

In [ ]:
# 4. 通知书备注分类函数
def classify_shangzhe(text):
    if pd.isna(text):
        return "否"
    text = str(text).strip()
    if text == "":
        return "否"

    # 否定模式（优先）
    if "商降" in text:
        return "否"
    if "商折不存在" in text:
        return "否"
    if re.search(r"商折\s*\d+\.?\d*\s*%", text):
        return "否"

    # 必须含商或返
    has_shang = "商" in text
    has_fan = "返" in text
    if not has_shang and not has_fan:
        return "否"

    # 返是强正向信号
    if has_fan:
        return "是"

    # 商 + 正向关键词
    positive_keywords = [
        "追回", "返还", "抵扣", "扣款",
        "签订商折", "追商折", "需要商折", "涉及商折",
        "存在商折", "商折抵扣", "一次性商折",
        "商折追回", "商折方式", "商折返还",
        "货款中抵扣", "开票中商折抵扣", "商折扣款",
        "月扣款", "月货款", "月份扣款",
        "商折",
    ]
    if has_shang:
        for kw in positive_keywords:
            if kw in text:
                return "是"
    return "否"

df["是否商折字段"] = df["通知书备注"].apply(classify_shangzhe)
print(f"商折分类统计:
{df["是否商折字段"].value_counts().to_string()}")

In [ ]:
# 5. 筛选是否商折字段 = 是
df_shangzhe = df[df["是否商折字段"] == "是"].copy()
print(f"商折记录数: {len(df_shangzhe)} 行")

In [ ]:
# 6. 保留所需字段
df_output = df_shangzhe[["价格通知书号", "供应商名称", "通知书备注", "创建人"]].copy()
df_output["识别日期"] = TODAY
df_output.rename(columns={"价格通知书号": "价格通知书"}, inplace=True)
df_output = df_output[["价格通知书", "供应商名称", "通知书备注", "创建人", "识别日期"]]

print(f"输出行数: {len(df_output)}")
df_output.head()

In [ ]:
# 7. 追加到累计文件
os.makedirs(OUTPUT_DIR, exist_ok=True)
try:
    existing = pd.read_excel(OUTPUT_FILE)
    # 去重（避免重复追加同一通知书）
    combined = pd.concat([existing, df_output], ignore_index=True)
    combined = combined.drop_duplicates(subset=["价格通知书"], keep="last")
    print(f"原有 {len(existing)} 行，新增 {len(df_output)} 行，去重后共 {len(combined)} 行")
except FileNotFoundError:
    combined = df_output
    print(f"新建文件，共 {len(combined)} 行")

combined.to_excel(OUTPUT_FILE, index=False)
print(f"已保存至: {OUTPUT_FILE}")